In [3]:
"""
JobFusion Application Script

Purpose:
--------
This script serves as the core functionality for the JobFusion application, which provides users with tools for managing their career-related documents and offers guidance on career development. 
The script is divided into three main functionalities:

1. Build Docs: provides updated resume and cover letter based on job postings. Users can upload their documents and input a job URL, 
and the script will generate tailored documents that align with the job qualifications.

2. Edit Docs: enables users to provide feedback on updated documents, then modifies existing resumes and cover letters based on feedback. 
The script processes user input to refine and enhance the documents, ensuring they are optimized for job qualifications and aligned with users' opinions.

3. Career Chat: offers a chatbot that assists users with mock interviews and answers career-related questions. 
The chatbot leverages natural language processing to provide personalized advice and guidance.

Overall, the script is designed to streamline the job application process and provide users with valuable insights and support in their career journey.

The advantages of this new version of the CareerAdvisor application include:
- Improved user interface and user experience
- Enabled document editing based on user feedback

environment: peotry install based on pyproject.toml; OpenAI API key is required

script usage: poetry run streamlit run final_jobfusion_app2.py

1st Version Date: 2024-08-18
"""
import os
import sys
import logging
import streamlit as st
import docx2txt
from crewai import Crew, Task, Agent, Process
from crewai_tools import ScrapeWebsiteTool
from jobfusion_agents import JobFusion_Agents
from jobfusion_tasks import JobFusion_Tasks
from jobfusion2_agents import JobFusion2_Agents
from jobfusion2_tasks import JobFusion2_Tasks
from langchain.chat_models import ChatOpenAI
from dotenv import load_dotenv
from Config import configure as cfg
from mock_interview_chatbot import *
import streamlit as st


/Users/frankwei/Documents/Side_Project/jobfusion_subj/jobfusion310/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 0.2.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(


In [4]:
from dotenv import load_dotenv
load_dotenv()
openai_api_key = os.getenv('OPENAI_API_KEY')
llm_35_turbo = ChatOpenAI(api_key=openai_api_key, model='gpt-3.5-turbo', temperature=0.7)
manager_llm_35_turbo = ChatOpenAI(api_key=openai_api_key, model='gpt-3.5-turbo')

class JobFusionCrew:
    def __init__(self, resume_input, personal_writeup_input, jd_url_input):
        self.resume_input = resume_input
        self.personal_writeup_input = personal_writeup_input  
        self.jd_url = jd_url_input 

    def run(self):
        # Initialize agents and tasks
        agents = JobFusion_Agents(self.resume_input, self.jd_url)
        tasks = JobFusion_Tasks(self.resume_input, self.personal_writeup_input, self.jd_url)

        crew = Crew(
            agents=[
                agents.researcher(), 
                agents.profiler(), 
                agents.resume_strategist(),
                agents.cover_letter_strategist(),
                agents.interview_preparer()
            ],
            tasks=[
                tasks.research_task(agents.researcher()),
                tasks.profile_task(agents.profiler()),
                tasks.resume_strategy_task(agents.resume_strategist()),
                tasks.cover_letter_strategy_task(agents.cover_letter_strategist()),
                tasks.interview_preparation_task(agents.interview_preparer())
            ],
            verbose=True
        )

        # Kickoff the process and return results
        results = crew.kickoff()
        return results

# JobFusionCrew 2 Class to modify the resume and cover letter based on users feedback
class JobFusionCrew2:
    def __init__(self, ori_resume_input, ori_personal_writeup_input, latest_resume_input, jd_qualifications_input, users_feedback):
        self.original_resume_input = ori_resume_input
        self.original_personal_writeup_input = ori_personal_writeup_input
        self.latest_resume_input = latest_resume_input
        self.jd_qualifications_input = jd_qualifications_input
        self.users_feedback = users_feedback

    def run(self):
        agents = JobFusion2_Agents(self.original_resume_input, self.original_personal_writeup_input, self.latest_resume_input)
        tasks = JobFusion2_Tasks(self.original_resume_input, self.original_personal_writeup_input, self.jd_qualifications_input, self.latest_resume_input, self.users_feedback)

        # Define the crew and hierarchical process
        crew = Crew(
            agents=[
                agents.resume_strategist(),
                agents.cover_letter_strategist(),
                agents.document_validation_manager()
            ],
            tasks=[
                tasks.resume_strategy_task(agents.resume_strategist()),
                tasks.cover_letter_strategy_task(agents.cover_letter_strategist()),
                tasks.document_validation_task(agents.document_validation_manager())
            ],
            process = Process.hierarchical, # Hierarchical process to manage delegation
            manager_llm = manager_llm_35_turbo,  # Assign the manager_llm to the Crew
            verbose=True
        )

        # Kickoff the process and return results
        results = crew.kickoff()
        return results

In [4]:
from jobfusion import create_resume_review_agent, cover_letter_strategist, interview_preparer


ValidationError: 1 validation error for ChatOpenAI
__root__
  Did not find openai_api_key, please add an environment variable `OPENAI_API_KEY` which contains it, or pass `openai_api_key` as a named parameter. (type=value_error)

In [6]:
resume_path = '/Users/frankwei/Documents/Side_Project/jobfusion_subj/JobFusion/inputs/resume_AI.docx'
personal_writeup_path = '/Users/frankwei/Documents/Side_Project/jobfusion_subj/JobFusion/inputs/skills_profile.docx'
jd_url_input = 'https://www.amazon.jobs/en/jobs/2558200/principal-applied-scientist'
JobFusionCrew(resume_path, personal_writeup_path, jd_url_input).run()

Inserting batches in chromadb:   0%|          | 0/1 [00:00<?, ?it/s]2024-10-14 17:35:07,791 - 8495755840 - local_persistent_hnsw.py-local_persistent_hnsw:271 - WARNING: Add of existing embedding ID: default-app-id--f84ac5ef2f0d12c93d726dff546949c0760e177683e2c042a1340960eeb67a7c
2024-10-14 17:35:07,793 - 8495755840 - local_persistent_hnsw.py-local_persistent_hnsw:271 - WARNING: Add of existing embedding ID: default-app-id--6fd78a686612a57f0518bd3b9becb7895ee735e48e470aa97b261b1bfb7d3cc4
2024-10-14 17:35:07,794 - 8495755840 - local_persistent_hnsw.py-local_persistent_hnsw:271 - WARNING: Add of existing embedding ID: default-app-id--6c8defe9a06e20a7ff1a8f385a75687adcf960cbccca4aab6d2432d67603bd14
2024-10-14 17:35:07,794 - 8495755840 - local_persistent_hnsw.py-local_persistent_hnsw:271 - WARNING: Add of existing embedding ID: default-app-id--d83bf957dd7e49b9ced3e2f8ae93412fda6af992a6ddc3814b4348b14b0dde26
2024-10-14 17:35:07,795 - 8495755840 - local_persistent_hnsw.py-local_persistent_hns

 [DEBUG]: == Working Agent: Data Scientist Job Researcher
 [INFO]: == Starting Task: 
Analyze the provided job description to extract key skills, experiences and qualifications required, based on 
the following given job descriptions URL.
If you do your BEST WORK, I'll tip you $100!

Job description URL: https://www.amazon.jobs/en/jobs/2558200/principal-applied-scientist



> Entering new CrewAgentExecutor chain...
I need to extract key skills, experiences, and qualifications from the job description URL provided. I should use the "Read website content" tool to gather the necessary information.

Action: Read website content
Action Input: {"url": "https://www.amazon.jobs/en/jobs/2558200/principal-applied-scientist"} 

Principal Applied Scientist - Job ID: 2558200 | Amazon.jobs
Skip to main contentHomeYour job applicationAmazon culture & benefitsDiversity at AmazonLocationsTeamsJob categoriesResourcesInterview tipsDisability accommodationsAbout AmazonFAQ×Principal Applied ScientistJob ID

"Interview Questions and Talking Points for Diana Liu:\n\n1. Can you walk us through your experience with implementing machine learning, optimization, and deep learning in your previous roles?\n2. How have you applied NLP techniques, specifically domain-specific word embedding and NER using BERT/GPT-3 transfer learning, in your projects at Fannie Mae?\n3. Could you provide an example of a successful project where you utilized Python, SQL, and AI technologies to tackle a complex business challenge?\n4. In your role as a Lead Data Scientist at The Gallup Organization, how did you lead predictive modeling projects for customer engagement? \n5. Can you discuss a specific instance where you collaborated with cross-functional teams to develop innovative solutions that delivered impactful results?\n6. How do you stay updated with the latest advancements in AI technologies and data science methodologies?\n7. What motivated you to pursue a Professional Degree in Artificial Intelligence from Sta

## To do list:
- Convert .md to .pdf
- Evaluation bot


In [6]:
# %pip install markdown2 pdfkit
# import markdown2
# import pdfkit
import markdown
import pdfkit

def md_to_pdf(md_file_path, pdf_file_path):
    # Read the Markdown file
    with open(md_file_path, 'r') as f:
        md_content = f.read()

    # Convert Markdown to HTML
    html_content = markdown.markdown(md_content, extensions=['extra'])
    # html_content = markdown.markdown(md_content, extensions=['extra', 'nl2br'])
    # Custom CSS to improve formatting
    custom_css = """
    <style>
        body { font-family: Arial, sans-serif; line-height: 1.6; }
        h1, h2, h3 { margin-top: 20px; }
        ul, ol { margin-left: 20px; padding-left: 0; }
        li { margin-bottom: 5px; }
    </style>
    """

    # Create a complete HTML document
    html_document = f"""
    <html>
    <head>
        <meta charset="utf-8">
        {custom_css}
    </head>
    <body>
        {html_content}
    </body>
    </html>
    """

    # Convert HTML to PDF
    options = {
        'page-size': 'Letter',
        'margin-top': '0.75in',
        'margin-right': '0.75in',
        'margin-bottom': '0.75in',
        'margin-left': '0.75in',
        'encoding': "UTF-8",
        'no-outline': None
    }
    pdfkit.from_string(html_document, pdf_file_path, options=options)

# Usage
md_to_pdf('output/updated_resume.md', 'output/updated_resume.pdf')

In [7]:
%pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 331.1 kB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.1.2 -> 24.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


you are a resume review expert. write a crew agent to review the modified resume, analyze the difference between the modified version and the original version, finally based on your experience, give the suggestion about what further needs to be done for the modified version.

write the script to generate the crew.ai agent python code

the original resume is in md format while the updated version is in pdf format

add a rate session to the crew agent to give a score of 1 to 10 to the modified version of the resume, and also give the reason about how the score is generated. 10 is perfect and there is no room for the improvement while 0 is a bad version that needs to be completely rewrote. 6 is the cutoff point that the modified version needs to be rewritten. please give the score based on criteria: 1. if the modified resume contains all the content in the original resume. 2. if the resume efficiently present all the information 3. if every section of the resume is clearly segmented out and good align with each other 4. if the resume will get HR's attention to pass HR's screen. 5. if the resume occupies the one entire page.

include the job description (jd, presented as a https link string) in the agent to make your suggestion is based on the link of jd as well

add a rewrite session to incorporate all the suggestion to come out with a new resume
Certainly! I'll add a rewrite session to incorporate all the suggestions and generate a new, improved resume. This will


In [ ]:
import os
import requests
from crewai import Agent, Task, Crew, Process
from langchain.llms import OpenAI
import markdown
import PyPDF2



In [13]:
# import os
# from crewai import Agent, Task, Crew, Process
# from langchain.llms import OpenAI
# import markdown
# import PyPDF2

# import os
# import requests
# from crewai import Agent, Task, Crew, Process
# from langchain.llms import OpenAI
# import markdown
# import PyPDF2

# def read_markdown_file(file_path):
#     with open(file_path, 'r', encoding='utf-8') as file:
#         md_content = file.read()
#     html_content = markdown.markdown(md_content)
#     return html_content

# def read_pdf_file(file_path):
#     with open(file_path, 'rb') as file:
#         pdf_reader = PyPDF2.PdfReader(file)
#         text_content = ""
#         for page in pdf_reader.pages:
#             text_content += page.extract_text()
#     return text_content

# def fetch_job_description(url):
#     response = requests.get(url)
#     if response.status_code == 200:
#         return response.text
#     else:
#         return f"Failed to fetch job description. Status code: {response.status_code}"

# def create_resume_review_agent():
#     resume_review_agent = Agent(
#         role='Resume Review Expert',
#         goal='Provide comprehensive and actionable feedback on resumes based on the job description',
#         backstory="""You are an experienced resume review expert with a keen eye for detail 
#         and a deep understanding of various industries and job markets. Your expertise 
#         helps job seekers create compelling resumes that stand out to potential employers and match job requirements.""",
#         verbose=True,
#         allow_delegation=False,
#         llm=OpenAI(temperature=0.7)
#     )

#     review_task = Task(
#         description="""Review the provided resumes and analyze their strengths and weaknesses. 
#         Focus on structure, content, formatting, and overall impact. Note that the original 
#         resume is in HTML format (converted from Markdown) and the updated resume is in plain text 
#         (extracted from PDF). Consider these format differences in your analysis. 
#         Compare the resumes against the provided job description to assess their relevance and fit.""",
#         agent=resume_review_agent,
#         expected_output="A detailed analysis of both resumes, highlighting strengths and weaknesses in relation to the job description."
#     )

#     compare_task = Task(
#         description="""Compare the modified resume with the original version. 
#         Identify and analyze the changes made and their potential impact. Be aware that 
#         formatting differences may exist due to the different file formats. 
#         Evaluate how well each version addresses the requirements in the job description.""",
#         agent=resume_review_agent,
#         expected_output="A comprehensive comparison of the two resumes, noting significant changes and their impacts, with reference to the job description."
#     )

#     suggest_task = Task(
#         description="""Based on your analysis and the job description, provide specific, actionable suggestions 
#         for further improving the resume. Consider industry best practices, current job market trends, 
#         and the specific requirements of the job. Address any issues that may have arisen from the conversion process.""",
#         agent=resume_review_agent,
#         expected_output="A list of specific, actionable suggestions for improving the resume to better match the job description."
#     )

#     rate_task = Task(
#         description="""Rate the modified resume on a scale of 1 to 10, where 10 is perfect with no room for improvement,
#         and 1 is a bad version that needs to be completely rewritten. A score of 6 is the cutoff point where the resume
#         needs significant revision. Base your rating on the following criteria:
#         1. If the modified resume contains all the relevant content from the original resume.
#         2. If the resume efficiently presents all the information.
#         3. If every section of the resume is clearly segmented and well-aligned with each other.
#         4. If the resume is likely to pass HR's initial screening.
#         5. If the resume occupies one entire page.
#         6. How well the resume matches the requirements in the job description.
#         Provide a detailed explanation for your score, addressing each criterion.""",
#         agent=resume_review_agent,
#         expected_output="A numerical score (1-10) for the modified resume with a detailed explanation of the rating based on the specified criteria and job description."
#     )

#     resume_review_crew = Crew(
#         agents=[resume_review_agent],
#         tasks=[review_task, compare_task, suggest_task, rate_task],
#         verbose=2,
#         process=Process.sequential
#     )

#     return resume_review_crew

# def run_resume_review(original_resume_path, modified_resume_path, jd_url):
#     original_resume = read_markdown_file(original_resume_path)
#     modified_resume = read_pdf_file(modified_resume_path)
#     job_description = fetch_job_description(jd_url)

#     crew = create_resume_review_agent()
#     result = crew.kickoff(
#         inputs={
#             "original_resume": original_resume,
#             "modified_resume": modified_resume,
#             "job_description": job_description
#         }
#     )
#     return result

# if __name__ == "__main__":
#     original_resume_path = "output/updated_resume.md"
#     modified_resume_path = "output/updated_resume.pdf"
#     job_description_url = "https://www.amazon.jobs/en/jobs/2558200/principal-applied-scientist"

#     review_result = run_resume_review(original_resume_path, modified_resume_path,job_description_url)
#     print(review_result)

In [23]:
import os
import requests
from crewai import Agent, Task, Crew, Process
from langchain.llms import OpenAI
import markdown
import PyPDF2

def read_markdown_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        md_content = file.read()
    html_content = markdown.markdown(md_content)
    return html_content

def read_pdf_file(file_path):
    with open(file_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        text_content = ""
        for page in pdf_reader.pages:
            text_content += page.extract_text()
    return text_content

def fetch_job_description(url):
    response = requests.get(url)
    if response.status_code == 200:
        return response.text
    else:
        return f"Failed to fetch job description. Status code: {response.status_code}"

def create_resume_review_agent():
    resume_review_agent = Agent(
        role='Resume Review Expert',
        goal='Provide comprehensive and actionable feedback on resumes based on the job description',
        backstory="""You are an experienced resume review expert with a keen eye for detail 
        and a deep understanding of various industries and job markets. Your expertise 
        helps job seekers create compelling resumes that stand out to potential employers and match job requirements.""",
        verbose=True,
        allow_delegation=False,
        llm=OpenAI(temperature=0.7)
    )

    review_task = Task(
        description="""Review the provided resumes and analyze their strengths and weaknesses. 
        Focus on structure, content, formatting, and overall impact. Note that the original 
        resume is in HTML format (converted from Markdown) and the updated resume is in plain text 
        (extracted from PDF). Consider these format differences in your analysis. 
        Compare the resumes against the provided job description to assess their relevance and fit.""",
        agent=resume_review_agent,
        expected_output="A detailed analysis of both resumes, highlighting strengths and weaknesses in relation to the job description."
    )

    compare_task = Task(
        description="""Compare the modified resume with the original version. 
        Identify and analyze the changes made and their potential impact. Be aware that 
        formatting differences may exist due to the different file formats. 
        Evaluate how well each version addresses the requirements in the job description.""",
        agent=resume_review_agent,
        expected_output="A comprehensive comparison of the two resumes, noting significant changes and their impacts, with reference to the job description."
    )

    suggest_task = Task(
        description="""Based on your analysis and the job description, provide specific, actionable suggestions 
        for further improving the resume. Consider industry best practices, current job market trends, 
        and the specific requirements of the job. Address any issues that may have arisen from the conversion process.""",
        agent=resume_review_agent,
        expected_output="A list of specific, actionable suggestions for improving the resume to better match the job description."
    )

    rate_task = Task(
        description="""Rate the modified resume on a scale of 1 to 10, where 10 is perfect with no room for improvement,
        and 1 is a bad version that needs to be completely rewritten. A score of 6 is the cutoff point where the resume
        needs significant revision. Base your rating on the following criteria:
        1. If the modified resume contains all the relevant content from the original resume.
        2. If the resume efficiently presents all the information.
        3. If every section of the resume is clearly segmented and well-aligned with each other.
        4. If the resume is likely to pass HR's initial screening.
        5. If the resume occupies one entire page.
        6. How well the resume matches the requirements in the job description.
        Provide a detailed explanation for your score, addressing each criterion.""",
        agent=resume_review_agent,
        expected_output="A numerical score (1-10) for the modified resume with a detailed explanation of the rating based on the specified criteria and job description."
    )

    reflect_task = Task(
        description="""Reflect on your review process and decision-making. Consider the following:
        1. What aspects of the resume review were most challenging?
        2. How did you balance the original content with the job description requirements?
        3. Were there any biases in your review process? How did you address or mitigate them?
        4. What assumptions did you make during the review, and how might they have affected your conclusions?
        5. If you were to do this review again, what would you do differently?
        6. How confident are you in your assessment and suggestions?
        Provide a thoughtful analysis of your own performance and thought process.""",
        agent=resume_review_agent,
        expected_output="A detailed self-reflection on the resume review process, including challenges faced, potential biases, and areas for improvement in the agent's approach."
    )

    resume_review_crew = Crew(
        agents=[resume_review_agent],
        tasks=[review_task, compare_task, suggest_task, rate_task, reflect_task],
        verbose=2,
        process=Process.sequential
    )

    return resume_review_crew

def run_resume_review(original_resume_path, modified_resume_path, jd_url):
    original_resume = read_markdown_file(original_resume_path)
    modified_resume = read_pdf_file(modified_resume_path)
    job_description = fetch_job_description(jd_url)

    crew = create_resume_review_agent()
    result = crew.kickoff(
        inputs={
            "original_resume": original_resume,
            "modified_resume": modified_resume,
            "job_description": job_description
        }
    )
    return result

if __name__ == "__main__":
    original_resume_path = "output/updated_resume.md"
    modified_resume_path = "output/updated_resume.pdf"
    job_description_url = "https://www.amazon.jobs/en/jobs/2558200/principal-applied-scientist"
    
    review_result = run_resume_review(original_resume_path, modified_resume_path, job_description_url)
    print(review_result)

PdfReadError: EOF marker not found

In [16]:
!pip install fpdf

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40704 sha256=3a24d6d537a7f5354cc95383325b41537b8e39742b4e4ea4b133aea45875a0ad
  Stored in directory: /Users/frankwei/Library/Caches/pip/wheels/f9/95/ba/f418094659025eb9611f17cbcaf2334236bf39a0c3453ea455
Successfully built fpdf

[notice] A new release of pip is available: 24.1.2 -> 24.2
[notice] To update, run: pip install --upgrade pip


In [ ]:
import os
import requests
from crewai import Agent, Task, Crew, Process
from langchain.llms import OpenAI
import markdown
import PyPDF2
from fpdf import FPDF

def read_markdown_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        md_content = file.read()
    html_content = markdown.markdown(md_content)
    return html_content

def read_text_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        txt_content = file.read()
    # html_content = markdown.markdown(md_content)
    return txt_content

def read_pdf_file(file_path):
    with open(file_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        text_content = ""
        for page in pdf_reader.pages:
            text_content += page.extract_text()
    return text_content

def fetch_job_description(url):
    response = requests.get(url)
    if response.status_code == 200:
        return response.text
    else:
        return f"Failed to fetch job description. Status code: {response.status_code}"

def markdown_to_pdf(markdown_content, output_path):
    # Convert Markdown to HTML
    html_content = markdown.markdown(markdown_content)

    # Create a PDF
    pdf = FPDF()
    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.set_font("Arial", size=12)

    # Split the HTML content into lines
    for line in html_content.split('\n'):
        if line.startswith('<h1>'):
            pdf.set_font("Arial", 'B', 16)
            pdf.cell(0, 10, line[4:-5], 0, 1)
            pdf.set_font("Arial", size=12)
        elif line.startswith('<h2>'):
            pdf.set_font("Arial", 'B', 14)
            pdf.cell(0, 10, line[4:-5], 0, 1)
            pdf.set_font("Arial", size=12)
        elif line.startswith('<p>'):
            pdf.multi_cell(0, 5, line[3:-4])
        elif line.startswith('<ul>') or line.startswith('</ul>'):
            continue
        elif line.startswith('<li>'):
            pdf.cell(10)
            pdf.multi_cell(0, 5, '• ' + line[4:-5])

    # Save the PDF
    pdf.output(output_path)

def create_resume_review_agent():
    resume_review_agent = Agent(
        role='Resume Review and Rewrite Expert',
        goal='Provide comprehensive feedback on resumes and rewrite them based on suggestions',
        backstory="""You are an experienced resume expert with a keen eye for detail 
        and a deep understanding of various industries and job markets. Your expertise 
        helps job seekers create compelling resumes that stand out to potential employers and match job requirements.""",
        verbose=True,
        allow_delegation=False,
        llm=OpenAI(temperature=0.7)
    )

    review_task = Task(
        description="""Review the provided resumes and analyze their strengths and weaknesses. 
        Focus on structure, content, formatting, and overall impact. Note that the original 
        resume is in HTML format (converted from Markdown) and the updated resume is in plain text 
        (extracted from PDF). Consider these format differences in your analysis. 
        Compare the resumes against the provided job description to assess their relevance and fit.""",
        agent=resume_review_agent,
        expected_output="A detailed analysis of both resumes, highlighting strengths and weaknesses in relation to the job description."
    )

    compare_task = Task(
        description="""Compare the modified resume with the original version. 
        Identify and analyze the changes made and their potential impact. Be aware that 
        formatting differences may exist due to the different file formats. 
        Evaluate how well each version addresses the requirements in the job description.""",
        agent=resume_review_agent,
        expected_output="A comprehensive comparison of the two resumes, noting significant changes and their impacts, with reference to the job description."
    )

    suggest_task = Task(
        description="""Based on your analysis and the job description, provide specific, actionable suggestions 
        for further improving the resume. Consider industry best practices, current job market trends, 
        and the specific requirements of the job. Address any issues that may have arisen from the conversion process.""",
        agent=resume_review_agent,
        expected_output="A list of specific, actionable suggestions for improving the resume to better match the job description."
    )

    rate_task = Task(
        description="""Rate the modified resume on a scale of 1 to 10, where 10 is perfect with no room for improvement,
        and 1 is a bad version that needs to be completely rewritten. A score of 6 is the cutoff point where the resume
        needs significant revision. Base your rating on the following criteria:
        1. If the modified resume contains all the relevant content from the original resume.
        2. If the resume efficiently presents all the information.
        3. If every section of the resume is clearly segmented and well-aligned with each other.
        4. If the resume is likely to pass HR's initial screening.
        5. If the resume occupies one entire page.
        6. How well the resume matches the requirements in the job description.
        Provide a detailed explanation for your score, addressing each criterion.""",
        agent=resume_review_agent,
        expected_output="A numerical score (1-10) for the modified resume with a detailed explanation of the rating based on the specified criteria and job description."
    )

    rewrite_task = Task(
        description="""Based on all the previous analysis, comparisons, suggestions, and ratings, 
        rewrite the resume to create an improved version. Ensure that the new version:
        1. Incorporates all the suggested improvements.
        2. Aligns closely with the job description requirements.
        3. Maintains a clear and professional structure.
        4. Highlights the candidate's most relevant skills and experiences.
        5. Is formatted for optimal readability and impact.
        Provide the rewritten resume in Markdown format, using appropriate Markdown syntax for headings, lists, and emphasis.""",
        agent=resume_review_agent,
        expected_output="A completely rewritten resume in Markdown format, incorporating all suggestions and improvements."
    )

    reflect_task = Task(
        description="""Reflect on your review and rewrite process. Consider the following:
        1. What were the most significant improvements made in the rewritten resume?
        2. How well does the new version address the job description requirements?
        3. What challenges did you face during the rewrite process?
        4. Are there any areas where you think further improvement might still be possible?
        5. How confident are you that this new version will significantly improve the candidate's chances?
        Provide a thoughtful analysis of the rewrite process and its outcomes.""",
        agent=resume_review_agent,
        expected_output="A detailed self-reflection on the resume rewrite process, including key improvements, challenges faced, and potential areas for further enhancement."
    )

    resume_review_crew = Crew(
        agents=[resume_review_agent],
        tasks=[review_task, compare_task, suggest_task, rate_task, rewrite_task, reflect_task],
        verbose=2,
        process=Process.sequential
    )

    return resume_review_crew

def run_resume_review(original_resume_path, modified_resume_path, jd_url, output_path):
    original_resume = read_markdown_file(original_resume_path)
    modified_resume = read_pdf_file(modified_resume_path)
    # job_description = fetch_job_description(jd_url)
    job_description = read_text_file(jd_url)
    crew = create_resume_review_agent()
    result = crew.kickoff(
        inputs={
            "original_resume": original_resume,
            "modified_resume": modified_resume,
            "job_description": job_description
        }
    )

    # Extract the rewritten resume from the result
    rewritten_resume = None
    for task_result in result:
        if "rewrite_task" in task_result:
            rewritten_resume = task_result["rewrite_task"]
            break

    if rewritten_resume:
        # Convert the Markdown content to PDF
        markdown_to_pdf(rewritten_resume, output_path)
        print(f"Rewritten resume saved as PDF to {output_path}")
    else:
        print("No rewritten resume found in the results.")
    return result


if __name__ == "__main__":
    original_resume_path = "output/updated_resume.md"
    modified_resume_path = "output/updated_resume.pdf"
    # job_description_url = "https://www.amazon.jobs/en/jobs/2558200/principal-applied-scientist"
    # job_description_url = 'https://www.google.com/about/careers/applications/jobs/results/129357347087622854-staff-machine-learning-software-engineer-ai-innovationresearch'
    job_description_url = 'output/jd.txt'
    output_resume_path = "output/rewritten_resume.pdf"
    
    review_result = run_resume_review(original_resume_path, modified_resume_path, job_description_url, output_resume_path)
    print(review_result)

MissingSchema: Invalid URL 'output/jd.txt': No scheme supplied. Perhaps you meant https://output/jd.txt?

In [ ]:
# create 

def create_resume_review_agent():
    resume_review_agent = Agent(
        role='Resume Review and Rewrite Expert',
        goal='Provide comprehensive feedback on resumes and rewrite them based on suggestions',
        backstory="""You are an experienced resume expert with a keen eye for detail 
        and a deep understanding of various industries and job markets. Your expertise 
        helps job seekers create compelling resumes that stand out to potential employers and match job requirements.""",
        verbose=True,
        allow_delegation=False,
        llm=OpenAI(temperature=0.7)
    )

    resume_review_agent = Agent(
        role='Resume Review and Rewrite Expert',
        goal='Provide comprehensive feedback on resumes and rewrite them based on suggestions',
        backstory="""You are an experienced resume expert with a keen eye for detail 
        and a deep understanding of various industries and job markets. Your expertise 
        helps job seekers create compelling resumes that stand out to potential employers and match job requirements.""",
        verbose=True,
        allow_delegation=False,
        llm=OpenAI(temperature=0.7)
    )

    review_task = Task(
        description="""Review the provided resumes and analyze their strengths and weaknesses. 
        Focus on structure, content, formatting, and overall impact. Note that the original 
        resume is in HTML format (converted from Markdown) and the updated resume is in plain text 
        (extracted from PDF). Consider these format differences in your analysis. 
        Compare the resumes against the provided job description to assess their relevance and fit.""",
        agent=resume_review_agent,
        output_file='output/jd.txt',
        expected_output="A detailed analysis of both resumes, highlighting strengths and weaknesses in relation to the job description."
    )


    resume_review_crew = Crew(
        agents=[resume_review_agent],
        tasks=[review_task, compare_task, suggest_task, rate_task, rewrite_task, reflect_task],
        verbose=2,
        process=Process.sequential
    )

    return resume_review_crew


def run_resume_review(original_resume_path, modified_resume_path, jd_url, output_path):
    original_resume = read_markdown_file(original_resume_path)
    modified_resume = read_pdf_file(modified_resume_path)
    # job_description = fetch_job_description(jd_url)
    job_description = read_text_file(jd_url)
    crew = create_resume_review_agent()
    result = crew.kickoff(
        inputs={
            "original_resume": original_resume,
            "modified_resume": modified_resume,
            "job_description": job_description
        }
    )

    # Extract the rewritten resume from the result
    rewritten_resume = None
    for task_result in result:
        if "rewrite_task" in task_result:
            rewritten_resume = task_result["rewrite_task"]
            break

    if rewritten_resume:
        # Convert the Markdown content to PDF
        markdown_to_pdf(rewritten_resume, output_path)
        print(f"Rewritten resume saved as PDF to {output_path}")
    else:
        print("No rewritten resume found in the results.")
    return result


if __name__ == "__main__":
    original_resume_path = "output/updated_resume.md"
    modified_resume_path = "output/updated_resume.pdf"
    # job_description_url = "https://www.amazon.jobs/en/jobs/2558200/principal-applied-scientist"
    # job_description_url = 'https://www.google.com/about/careers/applications/jobs/results/129357347087622854-staff-machine-learning-software-engineer-ai-innovationresearch'
    job_description_url = 'output/jd.txt'
    output_resume_path = "output/rewritten_resume.pdf"
    
    review_result = run_resume_review(original_resume_path, modified_resume_path, job_description_url, output_resume_path)
    print(review_result)

In [20]:
print(review_result)

After reviewing and rewriting the resume for John Doe, I am confident that the new version significantly improves his chances of securing a challenging role in project management or data analysis. My process involved a thorough review of the original resume, considering the job description requirements, and providing comprehensive feedback and suggestions for improvement.

The most significant improvements made in the rewritten resume include a more concise and impactful summary, highlighting John's key skills and experience. Additionally, the work experience section was revamped to focus on his achievements and results, rather than just listing job responsibilities. This makes the resume more compelling and impressive to potential employers.

The new version also addresses the job description requirements more effectively. I made sure to highlight John's experience in project management and data analysis, as well as his track record of successfully leading cross-functional teams and u

In [ ]:
def setup_streamlit_ui():
    st.write('Please upload your personal write-up, resume, and job description link.')
    st.text('')

    uploaded_resume = st.file_uploader('Step 1: Upload your resume:', type=['txt', 'docx', 'pdf'])
    uploaded_personal_writeup = st.file_uploader('Step 2: Upload your personal writeup:', type=['txt', 'docx', 'pdf'])
    jd_url_input = st.text_area(label='Step 3: Enter the URL of the job you are applying for:', placeholder='Paste the job description link/URL here...')

    return uploaded_resume, uploaded_personal_writeup, jd_url_input

# Main function to handle the application logic
def main():
    st.subheader('Welcome to JobFusion Crew !')
    st.write('**An AI-powered career advisory service with an agentic workflow to help you secure job interviews.**')
    
    tab1, tab2, tab3 = st.tabs(["Build Docs", "Edit Docs", "Career Chat"])

    # Tab 1: Build Documents (Resume and Cover Letter Creation)
    with tab1:
        uploaded_resume, uploaded_personal_writeup, jd_url_input = setup_streamlit_ui()
        if st.button('Start Processing'):
            if uploaded_resume and uploaded_personal_writeup and jd_url_input:
                st.write('Processing now....')
                logger.debug('Starting Agentic Workflow')

                # create output directory to store angent outputs
                os.makedirs("output/", exist_ok=True)

                # Save uploaded files
                resume_path = os.path.join("streamlit/", uploaded_resume.name)
                os.makedirs(os.path.dirname(resume_path), exist_ok=True)
                with open(resume_path, "wb") as f:
                    f.write(uploaded_resume.getbuffer())

                personal_writeup_path = os.path.join("streamlit/", uploaded_personal_writeup.name)
                os.makedirs(os.path.dirname(personal_writeup_path), exist_ok=True)
                with open(personal_writeup_path, "wb") as f:
                    f.write(uploaded_personal_writeup.getbuffer())

                # Run the JobFusionCrew process
                JobFusionCrew(resume_path, personal_writeup_path, jd_url_input).run()
                logger.debug('Agentic Workflow finished')
            else:
                st.error("Please upload both resume and personal writeup.")
                logger.error("Both resume and personal writeup are required.")

        col1, col2, col3 = st.columns([1, 1, 1])

        # Generate and download updated documents
        if col1.button('1 - Generate Resume'):
            with open('output/updated_resume.md', 'r') as file:
                resume_output = file.read()
            st.download_button('Download Resume', resume_output, file_name='updated_resume.txt', mime='text/plain')

        if col2.button('2 - Generate Cover Letter'):
            with open('output/coverletter.md', 'r') as file:
                cover_letter_output = file.read()
            st.download_button('Download Cover Letter', cover_letter_output, file_name='coverletter.txt', mime='text/plain')

        if col3.button('3 - Generate Interview Preparation Materials'):
            with open('output/interview_preparation_materials.txt', 'r') as file:
                interview_preparation_output = file.read()
            st.download_button('Download Interview Preparation', interview_preparation_output, file_name='interview_preparation_materials.txt', mime='text/plain')
    

    # Tab 2: Edit Documents (Resume and Cover Letter Modification based on Feedback)
    with tab2:
        st.subheader('Resume and Cover Letter Modification based on Your Feedback')

        # Gather user feedback for document modifications from four aspects
        missing_info = st.text_area(label='1. Which experiences or skills from your original resume do you believe were missed in the updated version?',
                                    placeholder="Please provide details here...")
        new_additions = st.text_area(label='2. Are there any additional achievements or project experiences that you would like to include in your resume?',
                                    placeholder="Please provide details here...")
        correct_inaccuracies = st.text_area(label='3. Do any parts of the updated resume or cover letter inaccurately represent your professional experience?',
                                    placeholder="Please provide details here...")
        general_suggestions = st.text_area(label='4. What additional suggestions do you have for improving the next version of your resume or cover letter?',
                                    placeholder="Please provide details here...")
        users_feedback = {
            'missing_info': missing_info,
            'new_additions': new_additions,
            'correct_inaccuracies': correct_inaccuracies,
            'general_suggestions': general_suggestions
        }
        
        if st.button('Start Modifying Documents Now'):
            st.write('Processing now....')
            logger.debug('Starting JobFusion2 Crew Agentic Workflow')

            # inputs paths for the modification process
            ori_resume_input = os.path.join("streamlit/", uploaded_resume.name)
            ori_personal_writeup_input = os.path.join("streamlit/", uploaded_personal_writeup.name)
            latest_resume_input = 'output/updated_resume.md'
            jd_qualifications_input = 'output/jd.txt'
            
            # Run the JobFusionCrew2 process
            JobFusionCrew2(ori_resume_input, ori_personal_writeup_input, latest_resume_input, jd_qualifications_input, users_feedback).run()
            logger.debug('Agentic Workflow JobFusionCrew2 finished')

        col1, col2 = st.columns([1, 1])

        # Generate and download revised documents
        if col1.button('1 - Generate Revised Resume'):
            with open('output/latest_resume.md', 'r') as file:
                resume_output = file.read()
            st.download_button('Download Revised Resume', resume_output, file_name='revised_resume.txt', mime='text/plain')

        if col2.button('2 - Generate Revised Cover Letter'):
            with open('output/latest_coverletter.md', 'r') as file:
                cover_letter_output = file.read()
            st.download_button('Download Revised Cover Letter', cover_letter_output, file_name='revised_coverletter.txt', mime='text/plain')

    # Tab 3: Career Advice/Mock Interview Chatbot
    with tab3:
        st.subheader('Career Advice Chatbot')
        st.write('You can chat with Career Adviser.')
        chat_container = st.container(height=300)

        # Ensure the necessary documents are uploaded
        if uploaded_resume and uploaded_personal_writeup and jd_url_input:
            logger.debug('Starting Career Advice Chatbot')

            # Load files and prepare vector database for chatbot
            files = file_loading("inputs/contents/")
            docs = doc_load_split(files)
            db = build_vectordb(docs)
            
            # Process user documents for chatbot context
            resume_path = os.path.join("streamlit/", uploaded_resume.name)
            personal_writeup_path = os.path.join("streamlit/", uploaded_personal_writeup.name)
            user_resume = docx2txt.process(resume_path)
            user_personal_writeup = docx2txt.process(personal_writeup_path)
            job_qualifications = []
            with open("output/jd.txt", 'r', encoding='utf-8') as file:
                content = file.read()
                job_qualifications.append(content)
            logger.debug("Resume and personal writeup loaded successfully.")

            # Initialize chat history if not already set
            if 'chat_history' in st.session_state:
                chat_history = st.session_state['chat_history']
            else:
                chat_history = []

            # Store generated responses
            if "messages" not in st.session_state.keys():
                st.session_state.messages = [{"role": "assistant", "content": "How may I help you?"}]

            # Display chat messages
            for message in st.session_state.messages:
                with chat_container.chat_message(message["role"]):
                    st.write(message["content"])
            
            # If vector database is ready, initialize the QA chain
            if db:
                chat_history = st.session_state.get('chat_history', [])
                qa_chain = get_qa_chain(
                    db, k=3, chain_type="stuff", user_resume=user_resume, 
                    user_personal_writeup=user_personal_writeup, job_qualifications=job_qualifications,
                    openai_api_key=openai_api_key, chat_history=chat_history
                )

                # Handle user input in the chat
                if prompt := st.chat_input("How can I help you?"):
                    if prompt is not None:
                        st.session_state.messages.append({'role': 'user', 'content': prompt})
                        with chat_container.chat_message('user'):
                            st.write(f'{prompt}')
                        with chat_container.chat_message('assistant'):
                            message_placeholder = st.empty()
                            full_response = ""
                            bot_response = qa_chain.run({"question": prompt, "chat_history": chat_history})
                            for chunk in re.findall(r'\S+|\n', bot_response):
                                full_response += chunk + " "
                                time.sleep(0.05)
                                message_placeholder.markdown(full_response + "▌")
                            message_placeholder.markdown(full_response)
                            chat_history = update_chat_history(chat_history, prompt, bot_response)

                            # Update chat history in session state
                            st.session_state['chat_history'] = chat_history
                            st.session_state.messages.append({'role': 'assistant', 'content': bot_response})
            else:
                st.error("Failed to build vector database.")
                logger.error("Failed to build vector database.")

            # End chat and collect feedback
            if 'chat_history' in st.session_state and st.button('Finish the Chat'):
                st.write("Thank you for using the Career Advisor chatbot! Have a great day!")
                feedback_text = st.text_area("Please provide feedback on your experience with the chatbot:")
                if st.button("Submit Feedback"):
                    st.write("Feedback submitted. Thank you!")
                    st.stop()

if __name__ == "__main__":
    main()
